In [1]:
"""
LangGraph 병렬 처리 + LLM 리포트 예제
- 병렬 노드:
  - fetch_weather,   - fetch_news,   - fetch_stocks
- 병렬 결과를 combine_results에서 합치고 마지막에 LLM(llm_report)이 사람 읽기좋은 자연어 리포트로 정리
- 병렬 노드들이 state 전체를 반환하지 않고, "자기 책임 키만" 부분 업데이트(Partial Update) 하도록 변경.
"""
!pip install langgraph langchain langchain-core langchain-community langchain-google-genai google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.1.3
    Uninstalling langchain-core-1.1.3:
      Successfully uninstalled langchain-core-1.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [3]:
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver  # 메모리에 상태 저장하는 체크포인터
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI( model="gemini-2.5-flash", temperature=0.3, )

In [4]:
# LangGraph 에서 사용할 전체 상태(State) 정의
class State(TypedDict, total=False):
    """
    LangGraph의 state는 "지금까지의 실행 결과"를 담는 큰 딕셔너리라고 보면 됨.
    total=False 이므로, 모든 키가 선택(optional)이다. (없어도 에러 아님)
    """
    # 입력 값(처음에만 넣고, 이후 노드에서는 읽기만 하는 값)
    location: str                   # 날씨/뉴스 지역
    symbols: List[str]              # 조회할 주식 종목 코드 리스트
    # 병렬 노드들이 채워줄 결과들
    weather: str                   # 날씨 결과 (fetch_weather 가 채움)
    news: List[str]                 # 뉴스 헤드라인 리스트 (fetch_news 가 채움)
    stocks: Dict[str, float]          # 종목별 주가 정보 (fetch_stocks 가 채움)

    # 중간 요약 / LLM용 텍스트
    summary: str                  # combine_results에서 만든 텍스트 요약 (LLM 입력으로 사용)

    # LLM 최종 자연어 리포트
    llm_report: str                 # llm_report 노드가 생성하는 최종 긴 리포트 텍스트

In [5]:

# 그래프의 각 노드 함수 정의
def init_state(state: State) -> dict:
    """
    초기 진입 노드. : 여기서는 "입력값이 제대로 들어왔나" 로그만 찍고,
    - 실제 state는 건드리지 않고 그대로 다음 노드들로 넘겨주기 위해 {} (빈 dict)를 리턴한다.
    """
    print(f"[init_state] 입력 location={state.get('location')}, symbols={state.get('symbols')}")
    # LangGraph의 "부분 업데이트" 규칙 때문에,
    # 여기서 state 전체를 반환하지 않고, 아무 것도 변경하지 않음을 의미하는 {}만 반환.
    return {}  # 입력 state는 그대로 유지되고, 변경 없음


def fetch_weather(state: State) -> dict:
    """
    날씨 정보 가져오기 (더미 데이터)
    - 실제로는 여기서 외부 Weather API를 호출하면 됨. state에서 location만 읽고, weather만 새로 채운다.
    """
    loc = state.get("location", "Seoul")  # location 이 없으면 기본값 "Seoul" 사용
    weather_text = f"{loc} : 맑고 가끔 구름, 23°C (더미 데이터)"
    print("[fetch_weather] 날씨 데이터 완료")
    # 이 노드는 weather 키만 업데이트. 나머지 키(location, news 등)는 건드리지 않기 때문에
    # 병렬 실행 시 충돌이 나지 않는다.
    return {"weather": weather_text}


def fetch_news(state: State) -> dict:
    """
    뉴스 정보 가져오기 (더미 데이터) : 실제로는 뉴스 API, RSS 등을 호출하면 됨.
    - 입력으로 받은 location을 참고해 간단한 더미 뉴스 목록 생성.
    """
    loc = state.get("location", "Korea")
    news_list = [
        f"{loc} 경제 성장률 상향 조정 (더미)",
        f"{loc} IT 기업, AI 투자 확대 (더미)",
        f"{loc} 증시, 외국인 매수세 유입 (더미)",
    ]
    print("[fetch_news] 뉴스 데이터 완료")
    return {"news": news_list}


def fetch_stocks(state: State) -> dict:
    """
    주식 정보 가져오기 (더미 데이터)
    - 실제로는 증권사/야후파이낸스 API를 호출해서 symbols 별 가격을 가져오면 됨.
    - 여기서는 간단히 dummy_prices 딕셔너리에서 찾아오는 방식으로 구현.
    """
    # 입력 state에서 symbols를 가져오고, 없으면 기본 종목 3개 사용
    symbols = state.get("symbols") or ["AAPL", "GOOG", "TSLA"]

    # 더미 가격 데이터 (실제에서는 API 결과가 여기에 들어오는 셈)
    dummy_prices = {
        "AAPL": 210.5,
        "GOOG": 135.2,
        "TSLA": 185.7,
        "NVDA": 125.3,
    }

    # 요청된 symbols 목록에 대해 가격을 매핑
    stocks = {sym: dummy_prices.get(sym, 100.0) for sym in symbols}
    print("[fetch_stocks] 주식 데이터 완료")
    return {"stocks": stocks}


def combine_results(state: State) -> dict:
    """
    병렬로 모은 날씨/뉴스/주식 결과를 하나의 summary 문자열로 합치는 노드.
    - 이 summary 텍스트는 사람이 읽을 수 있는 간단한 요약이고,
    - 다음 llm_report 노드에서 LLM에게 전달할 "입력 프롬프트의 재료"가 된다.
    """
    # 병렬 노드들이 채운 값들을 읽어옴. 없는 경우를 대비해 기본 문자열/빈 리스트 사용
    weather = state.get("weather", "날씨 정보 없음")
    news = state.get("news", [])
    stocks = state.get("stocks", {})

    lines = []
    lines.append("오늘의 종합 브리핑 (요약 데이터) ===")
    lines.append(f"[날씨]\n- {weather}")
    lines.append("[뉴스]")
    if news:
        # 뉴스 목록이 있으면 한 줄씩 예쁘게 출력
        for i, n in enumerate(news, start=1):
            lines.append(f"  {i}. {n}")
    else:
        lines.append("  - 뉴스 없음")

    lines.append("[주식]")
    if stocks:
        for sym, price in stocks.items():   # stocks 딕셔너리에서 (종목, 가격) 쌍을 한 줄씩 표시
            lines.append(f"  - {sym}: {price} USD (더미)")
    else:
        lines.append("  - 주식 데이터 없음")

    # 위에서 만든 줄들을 하나의 큰 문자열로 합침
    summary_text = "\n".join(lines)
    print("[combine_results] summary 생성 완료")
    # summary 키만 업데이트
    return {"summary": summary_text}


def llm_report(state: State) -> dict:
    """
    마지막 노드.
    - 앞에서 만든 summary 와 원본 데이터(weather, news, stocks)를 합쳐서
      LLM에게 "사람이 읽기 좋은 리포트 형태로 써달라"는 프롬프트를 보내고,
    - 그 결과를 llm_report 에 저장한다.
    """
    # 지금까지 state에 쌓인 데이터들을 읽어옴
    weather = state.get("weather", "")
    news = state.get("news", [])
    stocks = state.get("stocks", {})
    summary = state.get("summary", "")

    # LLM에게 전달할 프롬프트 문자열 구성
    prompt = (
        "당신은 친절한 경제/시황 리포트 작성자입니다.\n"
        "아래에 오늘의 날씨, 뉴스, 주식 데이터가 요약되어 있습니다.\n"
        "이를 바탕으로 일반인이 이해하기 쉬운 한국어 브리포트(3~5문단 정도)로 작성해 주세요.\n"
        "중요 트렌드, 투자자 관점에서 눈여겨 볼 포인트도 같이 정리해 주세요.\n\n"
        "원시 요약 데이터 ===\n"
        f"{summary}\n\n"
        "구조화된 데이터 ===\n"
        f"- 날씨: {weather}\n"
        f"- 뉴스 목록: {news}\n"
        f"- 주가 정보: {stocks}\n\n"
        "이 정보를 모두 참고해서 자연스럽게 서술형 기사 형태로 작성해 주세요."
    )

    print("[llm_report] LLM 호출 시작")
    # LangChain ChatModel 인터페이스: .invoke(prompt)를 호출하면
    # AIMessage 객체가 반환되고, resp.content 에 실제 텍스트가 들어 있음
    resp = llm.invoke(prompt)
    report_text = str(resp.content)  # 안전하게 문자열로 변환

    print("[llm_report] LLM 리포트 생성 완료")
    return {"llm_report": report_text}

In [7]:
# 그래프 구성 (노드 연결)
def build_app():
    """
    전체 LangGraph 실행 그래프를 구성하는 함수.
    구조:
    START → init
      → (fetch_weather, fetch_news, fetch_stocks)   ← 병렬 실행 가능한 구간
      → combine_results
      → llm_report
      → END
    """
    graph = StateGraph(State)  # 위에서 정의한 State 타입을 사용하는 그래프 생성

    # 1) 노드 등록: 이름("init" 등)과 실제 파이썬 함수 연결
    graph.add_node("init", init_state)
    graph.add_node("fetch_weather", fetch_weather)
    graph.add_node("fetch_news", fetch_news)
    graph.add_node("fetch_stocks", fetch_stocks)
    graph.add_node("combine_results", combine_results)
    graph.add_node("llm_report", llm_report)

    # 2) 노드 사이의 흐름(엣지) 정의
    # START → init : 그래프의 시작점에서 init 노드로 진입
    graph.add_edge(START, "init")

    # init 이후 세 개의 작업을 "병렬적으로" 실행하는 구조 (fan-out)
    # 이 세 노드는 모두 선행 조건이 init 하나뿐이라, 의존성이 없는 병렬 브랜치로 취급된다.
    graph.add_edge("init", "fetch_weather")
    graph.add_edge("init", "fetch_news")
    graph.add_edge("init", "fetch_stocks")

    # fetch_* 세 노드가 모두 끝나면 combine_results로 모이는 구조 (fan-in)
    graph.add_edge("fetch_weather", "combine_results")
    graph.add_edge("fetch_news", "combine_results")
    graph.add_edge("fetch_stocks", "combine_results")

    # 요약 이후, LLM이 자연어 리포트 생성
    graph.add_edge("combine_results", "llm_report")
    graph.add_edge("llm_report", END)  # 마지막 노드에서 END로 종료

    # 3) 체크포인터 설정
    # MemorySaver()는 state(상태)를 메모리에 저장/복원해 주는 간단한 체크포인터.
    # 같은 thread_id로 여러 번 실행하면 "이어서 실행" 같은 패턴도 만들 수 있다.
    checkpointer = MemorySaver()

    # graph.compile(...) 을 하면 "실행 가능한" app 이 만들어진다.
    app = graph.compile(checkpointer=checkpointer)
    return app     # 나중에 app.invoke / app.stream 으로 실행

In [8]:
if __name__ == "__main__":  # 이 파일을 직접 실행할 때만 아래 코드가 동작
    app = build_app()       # 위에서 구성한 LangGraph 앱 생성

    # thread_id는 "요청/세션" 단위를 구분하는 키
    # - 같은 thread_id로 실행하면, 체크포인터 기준으로 같은 세션으로 묶어서 관리할 수 있다.
    config = {
        "configurable": {
            "thread_id": "wns-llm-1"  # 아무 문자열이나 가능. 세션을 구분하는 ID 역할
        }
    }

    # 초기 입력 상태: location 과 symbols 만 넣어주면 나머지는 그래프가 채운다.
    init_state_value: State = {
        "location": "Seoul",                    # 날씨/뉴스 기준 지역
        "symbols": ["AAPL", "TSLA", "NVDA"],  # 보고 싶은 종목 코드
    }

    # 1) 전체 실행: 모든 노드를 순서대로(병렬 가능한 곳은 병렬로) 실행하고 최종 state 반환
    final_state: State = app.invoke(init_state_value, config=config)

    print("\n\n[최종 summary] ===")
    print(final_state.get("summary"))      # combine_results 가 만든 요약 텍스트 출력

    print("\n\n[LLM 자연어 리포트] ===")
    print(final_state.get("llm_report"))   # llm_report 가 만든 LLM 결과 출력

    # 2) stream() 으로 노드별 진행 상황 보기 (옵션)
    print("\n\nstream()으로 노드별 진행 상태 보기 ===")
    for step in app.stream( init_state_value,
        config={"configurable": {"thread_id": "stream-demo-llm"}},
        stream_mode="values",       # 각 단계별 state 전체를 dict 형태로 보내 줌
    ):
        # step 은 "그 시점까지의 state" 이므로 어떤 키들이 채워졌는지 확인 가능
        print("\n[stream step] 현재 state 키:", list(step.keys()))

[init_state] 입력 location=Seoul, symbols=['AAPL', 'TSLA', 'NVDA']
[fetch_news] 뉴스 데이터 완료
[fetch_stocks] 주식 데이터 완료
[fetch_weather] 날씨 데이터 완료
[combine_results] summary 생성 완료
[llm_report] LLM 호출 시작
[llm_report] LLM 리포트 생성 완료


[최종 summary] ===
오늘의 종합 브리핑 (요약 데이터) ===
[날씨]
- Seoul : 맑고 가끔 구름, 23°C (더미 데이터)
[뉴스]
  1. Seoul 경제 성장률 상향 조정 (더미)
  2. Seoul IT 기업, AI 투자 확대 (더미)
  3. Seoul 증시, 외국인 매수세 유입 (더미)
[주식]
  - AAPL: 210.5 USD (더미)
  - TSLA: 185.7 USD (더미)
  - NVDA: 125.3 USD (더미)


[LLM 자연어 리포트] ===
안녕하세요, 친애하는 투자자 여러분! 기분 좋은 하루 시작하셨나요?

오늘 서울은 맑고 가끔 구름이 끼는 쾌청한 날씨 속에 23°C를 기록하며 활동하기 좋은 하루가 예상됩니다. 경제와 증시도 이처럼 화창한 분위기를 이어갈 수 있을지 함께 살펴보시죠.

오늘 발표된 소식들을 보면, 서울 경제에 대한 기대감이 커지고 있습니다. 가장 먼저 눈에 띄는 것은 서울 경제 성장률이 상향 조정되었다는 소식입니다. 이는 전반적인 경기 회복세가 더욱 뚜렷해지고 있음을 시사하며, 기업 실적 개선에도 긍정적인 영향을 미칠 것으로 보입니다. 또한, 서울 증시에는 외국인 투자자들의 매수세가 꾸준히 유입되고 있다는 소식도 들려와 시장에 활력을 불어넣고 추가 상승 모멘텀을 제공할 수 있는 중요한 신호로 해석됩니다.

이러한 긍정적인 흐름 속에서 특정 산업의 약진도 주목할 만합니다. 특히 서울의 IT 기업들이 인공지능(AI) 분야 투자를 확대하고 있다는 소식은 미래 성장 동력에 대한 기대감을 높이고 있습니다. A